In [1]:
%pip install "psycopg[binary]" sqlalchemy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.8/4.8 MB 651.5 kB/s  0:00:078.2 kB/s eta 0:00:01:02
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [psycopg]

[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [1]:
import pandas as pd

clean_df = pd.read_csv("clean_work_orders.csv")

print(clean_df.shape)
clean_df.head()

(1536, 7)


,city_name,city_code,year,month,activity,status,work_order_count
0,شهر 1,6010,1401,7,test_1,in_progress,0
1,شهر 1,6010,1401,7,test_1,statement_preparation,0
2,شهر 1,6010,1401,7,test_1,at_headquarters,0
3,شهر 1,6010,1401,7,test_1,at_finance,0
4,شهر 1,6010,1401,7,test_2,in_progress,0


In [3]:
import psycopg
from getpass import getpass

db_password = getpass("Enter PostgreSQL password: ")

connection = psycopg.connect(
    host="localhost",
    port=5432,
    dbname="source_db",
    user="vaez_etl",
    password=db_password
)

print("Connected successfully!")

Enter PostgreSQL password:  ········


Connected successfully!


In [4]:
with connection.cursor() as cursor:
    cursor.execute("""
        SELECT
            current_database(),
            current_user,
            version();
    """)

    result = cursor.fetchone()

print("Database:", result[0])
print("User:", result[1])
print("Version:", result[2])

Database: source_db
User: vaez_etl
Version: PostgreSQL 14.18 (Homebrew) on aarch64-apple-darwin24.4.0, compiled by Apple clang version 17.0.0 (clang-1700.0.13.3), 64-bit


In [5]:
connection.close()

In [6]:
clean_df.columns.tolist()

['city_name',
 'city_code',
 'year',
 'month',
 'activity',
 'status',
 'work_order_count']

In [7]:
source_df = clean_df.rename(
    columns={
        "year": "report_year",
        "month": "report_month"
    }
).copy()

In [9]:
source_df.head()

,city_name,city_code,report_year,report_month,activity,status,work_order_count
0,شهر 1,6010,1401,7,test_1,in_progress,0
1,شهر 1,6010,1401,7,test_1,statement_preparation,0
2,شهر 1,6010,1401,7,test_1,at_headquarters,0
3,شهر 1,6010,1401,7,test_1,at_finance,0
4,شهر 1,6010,1401,7,test_2,in_progress,0


In [10]:
source_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1536 entries, 0 to 1535
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   city_name         1536 non-null   object
 1   city_code         1536 non-null   int64 
 2   report_year       1536 non-null   int64 
 3   report_month      1536 non-null   int64 
 4   activity          1536 non-null   object
 5   status            1536 non-null   object
 6   work_order_count  1536 non-null   int64 
dtypes: int64(4), object(3)
memory usage: 84.1+ KB


In [19]:
connection = psycopg.connect(
    host="localhost",
    port=5432,
    dbname="source_db",
    user="vaez_etl",
    password=db_password
)

In [14]:
insert_query = """
    INSERT INTO source_work_orders (
        city_name,
        city_code,
        report_year,
        report_month,
        activity,
        status,
        work_order_count
    )
    VALUES (%s, %s, %s, %s, %s, %s, %s)
    ON CONFLICT (
        city_code,
        report_year,
        report_month,
        activity,
        status
    )
    DO UPDATE SET
        city_name = EXCLUDED.city_name,
        work_order_count = EXCLUDED.work_order_count,
        source_file = 'clean_work_orders.csv',
        loaded_at = CURRENT_TIMESTAMP;
"""

In [15]:
records = list(
    source_df[
        [
            "city_name",
            "city_code",
            "report_year",
            "report_month",
            "activity",
            "status",
            "work_order_count"
        ]
    ].itertuples(index=False, name=None)
)

In [16]:
try:
    with connection.cursor() as cursor:
        cursor.executemany(insert_query, records)

    connection.commit()
    print(f"{len(records)} records loaded successfully.")

except Exception as error:
    connection.rollback()
    print("Loading failed:", error)

finally:
    connection.close()

1536 records loaded successfully.


In [20]:

with connection.cursor() as cursor:
    cursor.execute("""
        SELECT COUNT(*)
        FROM source_work_orders;
    """)

    row_count = cursor.fetchone()[0]

print("Database row count:", row_count)



Database row count: 1536
